# DSFB Unreal-Native Evidence Notebook

This notebook runs the strict Unreal-native replay path for `dsfb-computer-graphics`.

Real Unreal-native input means the buffers and metadata came from Unreal Engine. Synthetic, semi-synthetic, or proxy data must not be relabeled as `unreal_native`.


## What This Notebook Does

- uses a strict `dsfb_unreal_native_v1` manifest
- runs `cargo run --release -- run-unreal-native ...`
- displays the executive sheet and primary boardroom panel inline
- exposes `Download PDF` and `Download ZIP` controls


In [ ]:
from pathlib import Path
import json
import subprocess

crate_dir = Path('/content/dsfb/crates/dsfb-computer-graphics')
manifest_path = crate_dir / 'examples' / 'unreal_native_capture_manifest.json'
output_root = crate_dir / 'generated' / 'unreal_native_runs'
print('Manifest:', manifest_path)
print('Output root:', output_root)


## Refusal Condition

If the manifest or metadata are not labeled `unreal_native`, stop. This notebook should refuse to mislabel synthetic data as Unreal-native.


In [ ]:
manifest = json.loads(manifest_path.read_text())
assert manifest['dataset_kind'] == 'unreal_native'
assert manifest['provenance_label'] == 'unreal_native'
assert manifest['engine']['real_engine_capture'] is True
subprocess.run([
    'cargo', 'run', '--release', '--', 'run-unreal-native',
    '--manifest', str(manifest_path),
    '--output', str(output_root),
], cwd=crate_dir, check=True)


In [ ]:
run_dirs = sorted(output_root.iterdir())
run_dir = run_dirs[-1]
notebook_manifest = json.loads((run_dir / 'notebook_manifest.json').read_text())
print('Run dir:', run_dir)
print('Executive sheet:', notebook_manifest['executive_sheet_file_name'])
print('PDF:', notebook_manifest['pdf_bundle_file_name'])
print('ZIP:', notebook_manifest['zip_bundle_file_name'])


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(run_dir / notebook_manifest['executive_sheet_file_name'])))
display(Image(filename=str(run_dir / notebook_manifest['primary_panel_file_name'])))


In [ ]:
import ipywidgets as widgets
from google.colab import files

def download_pdf(_):
    files.download(str(run_dir / notebook_manifest['pdf_bundle_file_name']))

def download_zip(_):
    files.download(str(run_dir / notebook_manifest['zip_bundle_file_name']))

pdf_button = widgets.Button(description='Download PDF')
zip_button = widgets.Button(description='Download ZIP')
pdf_button.on_click(download_pdf)
zip_button.on_click(download_zip)
display(widgets.HBox([pdf_button, zip_button]))
